# 00 — Setup and data preparation

This notebook verifies that the working environment is properly configured for the ProvSAGE experiments on DARPA TRACE.

## What is checked
- Python and OS version
- GPU availability (NVIDIA A100 MIG slice expected)
- Key dependencies (`torch`, `torch_geometric`, `gensim`, `transformers`, ...)
- Free disk space
- Presence of the dataset, code, embedding models, and trained weights

## What this notebook does NOT do
This notebook does not download the dataset or train any model. It assumes the working tree at `/home/jovyan/stgan/work/STGAN-master/` is already populated with code, data, trained weights, and embedding models. Initial setup (Kaggle download, dependency install, data parsing) is documented in the project README.

## Next
Once all checks pass, proceed to `01_baseline_reproduction.ipynb`.

In [1]:
import os
from pathlib import Path

# Pour le reproducteur : adapter cette ligne au dossier où le dépôt est extrait.
# Sur la station ESTIN durant le développement : "/home/jovyan/stgan/work/STGAN-master"
PROJECT_ROOT = Path(os.environ.get("STGAN_ROOT", Path.cwd().resolve().parent))
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT incorrect : {PROJECT_ROOT}"
print(f"PROJECT_ROOT = {PROJECT_ROOT}\n")

import sys, platform
print(f"Python : {sys.version.split()[0]}")
print(f"OS : {platform.platform()}\n")

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA dispo : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go\n")

for pkg in ["torch_geometric", "gensim", "transformers", "numpy", "pandas", "sklearn", "tqdm"]:
    try:
        m = __import__(pkg)
        print(f"OK  {pkg:20s} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"--  {pkg:20s} (absent)")

import shutil
total, used, free = shutil.disk_usage(str(PROJECT_ROOT))
print(f"\nDisque sur PROJECT_ROOT : {free / 1e9:.1f} Go libres sur {total / 1e9:.1f} Go")

PROJECT_ROOT = /home/jovyan/stgan/work/STGAN-master

Python : 3.11.9
OS : Linux-6.8.0-110-generic-x86_64-with-glibc2.35

PyTorch : 2.6.0+cu124
CUDA dispo : True
GPU : NVIDIA A100 80GB PCIe MIG 1g.20gb
VRAM : 20.9 Go

OK  torch_geometric      2.7.0
OK  gensim               4.4.0
OK  transformers         5.9.0
OK  numpy                1.24.4
OK  pandas               2.1.4
OK  sklearn              1.3.2
OK  tqdm                 4.66.4

Disque sur PROJECT_ROOT : 8.1 Go libres sur 52.5 Go


In [ ]:
print("=== Verification : data + code + embedding models + weights ===\n")

checks = {
    "Data train (TRACE)"                       : PROJECT_ROOT / "data/trace/trace_train.txt",
    "Data test (TRACE)"                        : PROJECT_ROOT / "data/trace/trace_test.txt",
    "Ground truth"                             : PROJECT_ROOT / "data/trace/trace.json",
    "Code train/"                              : PROJECT_ROOT / "code/train",
    "Code eval/"                               : PROJECT_ROOT / "code/eval",
    "Code model/"                              : PROJECT_ROOT / "code/model",
    "Code utils/"                              : PROJECT_ROOT / "code/utils",
    "Word2Vec model"                           : PROJECT_ROOT / "embedding_models/word2vec_trace.model",
    "FastText model"                           : PROJECT_ROOT / "embedding_models/fasttext_trace.model",
    "Weights ProvSAGE (c2_w2v_graphsage_mha)"  : PROJECT_ROOT / "trained_weights/trace/c2_w2v_graphsage_mha",
    "Weights C2 cross-check (c2_graphsage_mha)": PROJECT_ROOT / "trained_weights/trace/c2_graphsage_mha",
}

for label, path in checks.items():
    if path.exists():
        if path.is_file():
            size = path.stat().st_size / 1e6
            print(f"OK  {label:45s} ({size:.1f} MB)")
        else:
            n_items = len(list(path.iterdir()))
            print(f"OK  {label:45s} ({n_items} fichiers)")
    else:
        print(f"--  {label:45s} ABSENT : {path}")